## Create Black masks file 

In [ ]:
import cv2
import numpy as np
import os

image_dir = '../data/figshare_braintumor/images/no_tumor/' 
mask_output_dir = '../data/figshare_braintumor/masks/no_tumor/'

os.makedirs(mask_output_dir, exist_ok=True)

for img_name in os.listdir(image_dir):
    img_path = os.path.join(image_dir, img_name)
    
    # Skip folders
    if os.path.isdir(img_path):
        continue
        
    # Create black mask & write 
    img = cv2.imread(img_path)
    if img is not None:
        black_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        cv2.imwrite(os.path.join(mask_output_dir, img_name), black_mask)

print(f"Success! Generated {len(os.listdir(mask_output_dir))} black masks.")

Success! Generated 1400 black masks.


## Change parse_filename

In [ ]:
import os
import shutil
from collections import defaultdict
from sklearn.model_selection import train_test_split

def parse_filename(filename):
    name, ext = os.path.splitext(filename)
    
    if ext.lower() not in ['.jpg', '.jpeg', '.png', '.tif']:
        return None
    
    # Handle Br35H "No Tumor" files
    if name.startswith('Tr-no') or name.startswith('no') or name.startswith('brisc2025'):
        return {
        'global_id': name,
        'patient_id': name,  # Treating each image as a single patient
        'label': 'no_tumor'
    }
        
    # Figshare files
    parts = name.split('_')
    if len(parts) >= 3:
        return {
            'global_id': parts[0],
            'patient_id': "_".join(parts[1:-1]),
            'label': parts[-1]
        }
    
    return None

patient_files = defaultdict(list)
skipped_files = [] 

images_root = "../data/figshare_braintumor/images"
masks_root  = "../data/figshare_braintumor/masks"

for label_folder in sorted(os.listdir(images_root)):          
    label_dir = os.path.join(images_root, label_folder)
    
    if not os.path.isdir(label_dir):
        continue
        
    for fname in os.listdir(label_dir):
        parsed = parse_filename(fname)
        if not parsed:
            skipped_files.append(f"FAILED PARSING: {fname}")
            continue
        
        # Extract values correctly from the dictionary
        global_num = parsed['global_id']
        patient_id = parsed['patient_id']
        label      = parsed['label']
        
        img_path  = os.path.join(images_root, label_folder, fname)
        mask_path = os.path.join(masks_root,  label_folder, fname)
        
        # Check if the mask exists
        if os.path.exists(mask_path):
            patient_files[patient_id].append((img_path, mask_path, label))
        else:
            skipped_files.append(f"MISSING MASK: {fname}")

total_valid_files = sum(len(v) for v in patient_files.values())
print(f"Successfully loaded {total_valid_files} valid image/mask pairs.")

if skipped_files:
    print(f"\nSkipped {len(skipped_files)} files. Here are a few examples:")
    for skipped in skipped_files[:10]:
        print(f" - {skipped}")

# Split patient IDs (stratify by majority label per patient)
if total_valid_files > 0:
    patient_ids = list(patient_files.keys())
    patient_labels = [patient_files[pid][0][2] for pid in patient_ids]

    train_pids, test_pids = train_test_split(
        patient_ids,
        test_size=0.2,
        stratify=patient_labels,   
        random_state=42
    )

    train_pids, val_pids = train_test_split(
        train_pids,
        test_size=0.1,
        stratify=[patient_files[pid][0][2] for pid in train_pids],
        random_state=42
    )

    print(f"\nPatients → Train: {len(train_pids)}, Val: {len(val_pids)}, Test: {len(test_pids)}")
    assert not set(train_pids) & set(test_pids), "LEAKAGE DETECTED"
    assert not set(val_pids)   & set(test_pids), "LEAKAGE DETECTED"

# Copy files into new directory
def copy_split(pid_list, split_name, base_out="../data/figshare_braintumor_split"):
    for pid in pid_list:
        for img_path, mask_path, label in patient_files[pid]:
            for src, kind in [(img_path, "images"), (mask_path, "masks")]:
                dst_dir = os.path.join(base_out, split_name, kind, label)
                os.makedirs(dst_dir, exist_ok=True)
                shutil.copy2(src, dst_dir)

print("\nCopying files...")
copy_split(train_pids, "train")
copy_split(val_pids,   "val")
copy_split(test_pids,  "test")
print("Done!")

Successfully loaded 4464 valid image/mask pairs.

Patients → Train: 1175, Val: 131, Test: 327

Copying files...
Done!
